# Ship driven into a waterfront building

5 m initial surge on a 12 m GeoClaw grid drives a 60 m ship toward the Seattle waterfront

In [ ]:
from pathlib import Path
import hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from netCDF4 import Dataset
from shapely import wkt
from shapely.geometry import box
from shapely.ops import unary_union, transform
sys.path.insert(0, str(Path('../../src').resolve()))
from geoclaw_debris.ship_tracking import Flow, Ship, simulate, hull_polygon

ds = Dataset('pike_place_surge5_fgout.nc')
assert ds.terrain_sha256 == hashlib.sha256(Path('seattle_buildings_mhw.tt3').read_bytes()).hexdigest()
assert ds.surge_height_m == 5.0 and ds.grid_spacing_m == 12.0
flow = Flow(ds)
extent = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
buildings = unary_union([p for p in (wkt.loads(s) for s in pd.read_csv('../../SeattleBuildings/Seattle_Inventory.csv').geometry) if p.intersects(extent)])

In [ ]:
ship = Ship()
start = (-122.3425, 47.6055)
track = pd.DataFrame(simulate(flow, buildings, ship, start))
net = np.hypot(track.x_m.iloc[-1]-track.x_m.iloc[0], track.y_m.iloc[-1]-track.y_m.iloc[0])
first_stop = track.loc[track.status != 'afloat'].iloc[0]
print(f'net drift: {net:.1f} m; first contact: {first_stop.status} at {first_stop.time_s:.0f} s; minimum depth: {first_stop.min_depth_m:.2f} m')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
origin = track[['x_m','y_m']].iloc[0].to_numpy()
ix = abs(flow.lon-start[0]) < .003
iy = abs(flow.lat-start[1]) < .003
lon, lat = np.meshgrid(flow.lon[ix], flow.lat[iy], indexing='ij')
x, y = flow.to_xy(lon, lat)
depth = np.asarray(ds['h'][12])[np.ix_(ix, iy)]
ax.pcolormesh(x-origin[0], y-origin[1], np.ma.masked_where(depth < .1, depth), cmap='Blues', vmin=0, vmax=30, shading='auto')
row = track.iloc[0]
outline = hull_polygon((row.x_m,row.y_m), 0., ship)
hull = PolygonPatch(np.asarray(outline.exterior.coords)-origin, facecolor='tomato', edgecolor='red', lw=2, alpha=.8)
building_xy = transform(flow.to_xy, buildings)
view = box(origin[0]-140, origin[1]-130, origin[0]+210, origin[1]+130)
for poly in (building_xy.geoms if hasattr(building_xy, 'geoms') else [building_xy]):
    if poly.intersects(view):
        ax.add_patch(PolygonPatch(np.asarray(poly.exterior.coords)-origin, facecolor='darkorange', edgecolor='saddlebrown', lw=0.7, alpha=0.75))
ax.add_patch(hull)
trail, = ax.plot([], [], 'r-', lw=2)
ax.set(xlim=(-140,210), ylim=(-130,130), xlabel='east (m)', ylabel='north (m)')
ax.set_aspect('equal')
ax.set_title('ship and waterfront buildings: 0 s, afloat')
fig.tight_layout()

def update(i):
    row = track.iloc[i]
    outline = hull_polygon((row.x_m,row.y_m), np.deg2rad(row.heading_deg), ship)
    hull.set_xy(np.asarray(outline.exterior.coords)-origin)
    trail.set_data(track.x_m.iloc[:i+1]-origin[0], track.y_m.iloc[:i+1]-origin[1])
    ax.set_title(f'ship and waterfront buildings: {int(row.time_s)} s, {row.status}')

anim = FuncAnimation(fig, update, frames=range(0,601,10), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Still frame at first contact
contact = first_stop
contact_xy = np.array([contact.x_m, contact.y_m])
contact_i = int(np.argmin(abs(flow.times-contact.time_s)))
contact_depth = np.asarray(ds['h'][contact_i])[np.ix_(ix, iy)]
fig_contact, ax_contact = plt.subplots(figsize=(8, 6))
ax_contact.pcolormesh(x-origin[0], y-origin[1], np.ma.masked_where(contact_depth < .1, contact_depth), cmap='Blues', vmin=0, vmax=30, shading='auto')
for poly in (building_xy.geoms if hasattr(building_xy, 'geoms') else [building_xy]):
    if poly.intersects(box(contact_xy[0]-110, contact_xy[1]-90, contact_xy[0]+110, contact_xy[1]+90)):
        ax_contact.add_patch(PolygonPatch(np.asarray(poly.exterior.coords)-origin, facecolor='darkorange', edgecolor='saddlebrown', lw=0.7, alpha=0.75))
ax_contact.plot(track.loc[track.time_s <= contact.time_s, 'x_m']-origin[0], track.loc[track.time_s <= contact.time_s, 'y_m']-origin[1], 'r-', lw=2)
contact_hull = hull_polygon(contact_xy, np.deg2rad(contact.heading_deg), ship)
ax_contact.add_patch(PolygonPatch(np.asarray(contact_hull.exterior.coords)-origin, facecolor='tomato', edgecolor='red', lw=2, alpha=.85))
ax_contact.set(xlim=(contact_xy[0]-origin[0]-100, contact_xy[0]-origin[0]+100), ylim=(contact_xy[1]-origin[1]-80, contact_xy[1]-origin[1]+80), xlabel='east of start (m)', ylabel='north of start (m)', title=f'Building contact at {contact.time_s:.0f} s; minimum depth {contact.min_depth_m:.2f} m')
ax_contact.set_aspect('equal')
fig_contact.tight_layout()
fig_contact
